In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel

In [ ]:
def char_counts(text: str) -> int:
    print(f"Calculating character count for: {text}\n")
    return len(text)

def word_counts(text: str) -> int:
    print(f"\nCalculating word count for: {text}\n")
    return len(text.split())

def summarize(info: dict) -> str:
    print(f"Summarizing info:")
    print ("Summary character count: " + str(info["char_count"]))
    print ("Summary word count: " + str(info["word_count"]))
    print ("Summary output: " + info["output"])
    return "Summary complete"

In [22]:
char_count_runnable = RunnableLambda(char_counts)
word_count_runnable = RunnableLambda(word_counts)
summarize_runnable = RunnableLambda(summarize)

prompt = ChatPromptTemplate.from_template("""Explain these inputs: {input1} and {input2} in 2 sentences.""")
llm = ChatOllama(
    model = "qwen3",
    base_url="http://localhost:11434"
)

chain = prompt | llm | StrOutputParser() | {
    "char_count": char_count_runnable, 
    "word_count": word_count_runnable,
    "output": RunnablePassthrough()
    } | summarize_runnable
output = chain.invoke({"input1": "Hello world", "input2": "How are you?"})
print(output)


Calculating character count for: "Hello world" is a common greeting used in programming to test code or introduce a new project, often serving as the first line of a program. "How are you?" is a standard question to inquire about someone's well-being or current state.

Calculating word count for: "Hello world" is a common greeting used in programming to test code or introduce a new project, often serving as the first line of a program. "How are you?" is a standard question to inquire about someone's well-being or current state.


Summarizing info:
Summary character count: 235
Summary word count: 41
Summary output: "Hello world" is a common greeting used in programming to test code or introduce a new project, often serving as the first line of a program. "How are you?" is a standard question to inquire about someone's well-being or current state.
Summary complete


In [23]:
#Another way to do the same thing is to use RunnableParallel which will run the char_count_runnable and word_count_runnable in parallel and return the results in a dictionary.

parallel = RunnableParallel({
    "char_count": char_count_runnable,
    "word_count": word_count_runnable,
    "output": RunnablePassthrough()
})
parallel_output = prompt |llm | StrOutputParser() | parallel | summarize_runnable
output = parallel_output.invoke({"input1": "Hello world", "input2": "How are you?"})
print(output)

Calculating character count for: "Hello world" is a common greeting used to initiate a conversation or express friendliness. "How are you?" is a question that asks about someone's current state of health or well-being.

Calculating word count for: "Hello world" is a common greeting used to initiate a conversation or express friendliness. "How are you?" is a question that asks about someone's current state of health or well-being.


Summarizing info:
Summary character count: 185
Summary word count: 30
Summary output: "Hello world" is a common greeting used to initiate a conversation or express friendliness. "How are you?" is a question that asks about someone's current state of health or well-being.
Summary complete


In [27]:
from langchain_core.runnables import chain

@chain
def parallel_chain(params):
    print(f"Running parallel chain with params: {params}")
    parallel = RunnableParallel({
        "char_count": char_count_runnable,
        "word_count": word_count_runnable,
        "output": RunnablePassthrough()
    })
    return parallel


parallel_output = prompt |llm | StrOutputParser() | parallel_chain | summarize_runnable
output = parallel_output.invoke({"input1": "Hello world", "input2": "How are you?"})
print(output)

Running parallel chain with params: "Hello world" is a common greeting used to test new programming languages or introduce basic syntax, often serving as a beginner's first program. "How are you?" is a standard question to inquire about someone's well-being or current state.
Calculating character count for: "Hello world" is a common greeting used to test new programming languages or introduce basic syntax, often serving as a beginner's first program. "How are you?" is a standard question to inquire about someone's well-being or current state.


Calculating word count for: "Hello world" is a common greeting used to test new programming languages or introduce basic syntax, often serving as a beginner's first program. "How are you?" is a standard question to inquire about someone's well-being or current state.

Summarizing info:
Summary character count: 239
Summary word count: 38
Summary output: "Hello world" is a common greeting used to test new programming languages or introduce basic s